In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE


df = pd.read_csv(r"C:\Users\Zahra\Downloads\autismdiagnosis\Autism_Prediction\train.csv")


X = df.drop('Class/ASD', axis=1)
y = df['Class/ASD']


numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'bool', 'category']).columns

print("Numeric features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))


numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
]) if len(categorical_cols) > 0 else None


transformers = []
if len(numerical_cols) > 0:
    transformers.append(('num', numerical_transformer, numerical_cols))
if len(categorical_cols) > 0:
    transformers.append(('cat', categorical_transformer, categorical_cols))

preprocessor = ColumnTransformer(transformers)

X_processed = preprocessor.fit_transform(X)

num_features = numerical_cols
cat_features = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else []
all_feature_names = np.concatenate([num_features, cat_features])


smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_processed, y)
print("✅ After SMOTE balancing:", X_balanced.shape, y_balanced.shape)


selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42))
selector.fit(X_balanced, y_balanced)
X_selected = selector.transform(X_balanced)
selected_indices = selector.get_support(indices=True)
selected_feature_names = all_feature_names[selected_indices]


X_clean = pd.DataFrame(X_selected, columns=selected_feature_names)
y_clean = pd.DataFrame(y_balanced, columns=['Class/ASD'])
cleaned_df = pd.concat([X_clean, y_clean], axis=1)

# ---- Step 8: Save cleaned dataset ----
cleaned_df.to_excel(r"C:\Users\Zahra\Downloads\autismdiagnosis\Autism_Prediction\cleaned_asd_dataset.xlsx", index=False)
print("✅ Clean dataset saved successfully as 'cleaned_asd_dataset.xlsx'!")

# ---- Step 9: Preview the cleaned data ----
cleaned_df.head()


Numeric features: 13
Categorical features: 8
✅ After SMOTE balancing: (1278, 96) (1278,)
✅ Clean dataset saved successfully as 'cleaned_asd_dataset.xlsx'!


,ID,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,result,ethnicity_?,ethnicity_White-European,austim_no,austim_yes,contry_of_res_United States,Class/ASD
0,-1.729887,0.886405,-1.061913,1.105542,-0.842260,1.237597,-0.660504,1.231147,-1.017656,1.010051,0.787041,0.596329,-0.455003,1.0,0.0,1.0,0.0,0.0,0
1,-1.725557,-1.128152,-1.061913,-0.904534,-0.842260,-0.808018,-0.660504,-0.812251,-1.017656,-0.990050,-1.270582,1.183895,-1.307503,1.0,0.0,1.0,0.0,0.0,0
2,-1.721227,0.886405,0.941697,1.105542,1.187282,1.237597,1.513995,1.231147,0.982650,1.010051,0.787041,-1.292684,1.314176,0.0,1.0,0.0,1.0,1.0,1
3,-1.716897,-1.128152,-1.061913,-0.904534,-0.842260,-0.808018,-0.660504,-0.812251,-1.017656,-0.990050,-1.270582,-0.299998,-1.303042,1.0,0.0,1.0,0.0,1.0,0
4,-1.712567,-1.128152,-1.061913,-0.904534,-0.842260,-0.808018,-0.660504,-0.812251,-1.017656,-0.990050,-1.270582,0.905091,-2.771177,1.0,0.0,1.0,0.0,0.0,0


  Using cached PyBluez-0.23.tar.gz (97 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [1 lines of output]
      error in PyBluez setup command: use_2to3 is invalid.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'bluetooth'